# Integración de datos meteorológicos históricos desde la API de OPEN-METEO
Open Meteo es una interfaz de programación de aplicaciones (API) y plataforma de código abierto que ofrece datos meteorológicos gratuitos, pronósticos del tiempo e información histórica a nivel global, para este proyecto, se extraerán todos los registros correspondientes al periodo 2000-2025 de la región Piura, exactamente en la estación Miraflores de la UNP.

## Importación de librerías
La librería request permitirá consumir la API de Open-Meteo para extraer todos los datos metereológicos en formato JSON, mientras que la librería pandas servirá para transformar el dataset JSON en un DataFrame manejable.

Finalmente la librería Path permitirá direccionar la exportación del dataset final hacia la carpeta Datasets de este proyecto.

In [26]:
from pathlib import Path
import requests
import pandas as pd

# 1. CONFIGURACIÓN DEL ENDPOINT Y PARÁMETROS (Estación Miraflores - Piura)
url_api = "https://archive-api.open-meteo.com/v1/archive"

## Descarga de datos climáticos
Los parámetros incluidos señalan el lugar exacto de la Estación Miraflores, el periodo de datos a extraer y las variables exactas que se necesitarán para este proyecto. Finalmente, estos datos se convertirán en un DataFrame con las columnas de Fecha, Temp_Max, Temp_Min y Precipitacion.

In [27]:
parametros = {
    "latitude": -5.1783,           # Coordenada exacta Estación Miraflores (UNP)
    "longitude": -80.6169,
    "start_date": "2000-01-01",
    "end_date": "2025-12-31",
    "daily": ["temperature_2m_max", "temperature_2m_min", "precipitation_sum"],
    "timezone": "America/Lima"
}

print("Descargando 26 años de datos climáticos diarios desde la API...")
respuesta = requests.get(url_api, params=parametros)
datos_json = respuesta.json()

# 2. ESTRUCTURACIÓN EN DATAFRAME DIARIO
df_clima_diario = pd.DataFrame({
    'Fecha': pd.to_datetime(datos_json['daily']['time']),
    'Temp_Max': datos_json['daily']['temperature_2m_max'],
    'Temp_Min': datos_json['daily']['temperature_2m_min'],
    'Precipitacion': datos_json['daily']['precipitation_sum']
})

print(f"Total de días descargados sin baches: {len(df_clima_diario)}")
display(df_clima_diario.head(10))

Descargando 26 años de datos climáticos diarios desde la API...
Total de días descargados sin baches: 9497


,Fecha,Temp_Max,Temp_Min,Precipitacion
0,2000-01-01,30.8,18.6,0.0
1,2000-01-02,31.4,19.8,0.0
2,2000-01-03,31.2,19.6,0.0
3,2000-01-04,33.0,18.8,0.0
4,2000-01-05,29.4,20.0,0.0
5,2000-01-06,32.1,19.8,0.1
6,2000-01-07,32.1,19.4,0.0
7,2000-01-08,31.8,20.4,0.0
8,2000-01-09,32.7,20.4,0.0
9,2000-01-10,32.1,19.6,0.0


# Agrupación de datos por año y semana
Para alinear los registros extraídos con el dataset del Minsa, se crean las columnas ano y semana para almacenar la semana y el año correspondiente a cada uno de los días extraídos.

In [28]:
# 3. CONVERSIÓN A SEMANA EPIDEMIOLÓGICA (Alineación con el MINSA: Domingo a Sábado)
domingo_epi = df_clima_diario['Fecha'] - pd.to_timedelta((df_clima_diario['Fecha'].dt.dayofweek + 1) % 7, unit='D')
miercoles_epi = domingo_epi + pd.Timedelta(days=3)
df_clima_diario['ano'] = miercoles_epi.dt.year.astype(int)
df_clima_diario['semana'] = ((miercoles_epi.dt.dayofyear - 1) // 7 + 1).astype(int)

# Filtramos posibles días residuales de finales de dic-1999 o inicios de 2026 según el calendario epidemiológico
df_clima_diario = df_clima_diario[(df_clima_diario['ano'] >= 2000) & (df_clima_diario['ano'] <= 2025)]

display(df_clima_diario.head(10))



,Fecha,Temp_Max,Temp_Min,Precipitacion,ano,semana
2,2000-01-03,31.2,19.6,0.0,2000,1
3,2000-01-04,33.0,18.8,0.0,2000,1
4,2000-01-05,29.4,20.0,0.0,2000,1
5,2000-01-06,32.1,19.8,0.1,2000,1
6,2000-01-07,32.1,19.4,0.0,2000,1
7,2000-01-08,31.8,20.4,0.0,2000,1
8,2000-01-09,32.7,20.4,0.0,2000,1
9,2000-01-10,32.1,19.6,0.0,2000,2
10,2000-01-11,32.5,19.3,0.0,2000,2
11,2000-01-12,32.5,19.5,0.0,2000,2


## Transformación de columnas metereológicas
Las columnas Precipitación y Temperatura serán transformadas para contener sumatorias y promedios, respectivamente, agrupados por la semana de cada año del periodo 2000-2025, encajando así con el Dataset transformado del MINSA y manteniendo el enfoque de series de tiempo semanales que son el objetivo de este proyecto.

In [29]:
# 4. AGREGACIÓN FÍSICA SEMANAL
# La lluvia se acumula (suma de charcos), las temperaturas se promedian
df_clima_semanal = df_clima_diario.groupby(['ano', 'semana']).agg({
    'Precipitacion': 'sum',
    'Temp_Max': 'mean',
    'Temp_Min': 'mean'
}).reset_index()

# Redondeamos a 2 decimales para limpiar el tensor
df_clima_semanal = df_clima_semanal.round(2)

print("\n--- VISTA PREVIA DEL CLIMA SEMANAL EN PIURA (2000-2025) ---")
print(df_clima_semanal.head(10))

# Guardamos un respaldo en CSV para no consultar la API cada vez que abras el notebook
ruta_datasets = next(
    (ruta / 'Datasets' for ruta in (Path.cwd(), *Path.cwd().parents)
     if (ruta / 'Datasets').is_dir()),
    None
)
if ruta_datasets is None:
    raise FileNotFoundError("No se encontró la carpeta 'Datasets' del proyecto.")

ruta_csv = ruta_datasets / 'clima_piura_2000_2025_semanal.csv'
df_clima_semanal.to_csv(ruta_csv, index=False)
print(f"Dataset guardado en: {ruta_csv}")


--- VISTA PREVIA DEL CLIMA SEMANAL EN PIURA (2000-2025) ---
    ano  semana  Precipitacion  Temp_Max  Temp_Min
0  2000       1            0.1     31.76     19.77
1  2000       2            0.0     32.07     19.73
2  2000       3            0.0     32.90     21.50
3  2000       4            8.6     31.79     22.54
4  2000       5           13.1     32.16     22.49
5  2000       6            2.6     33.10     22.37
6  2000       7            8.3     33.54     22.47
7  2000       8           22.6     31.54     22.73
8  2000       9            7.7     31.07     22.86
9  2000      10           27.0     33.11     23.26
Dataset guardado en: c:\Users\fabri\OneDrive\Desktop\PROYECTO-IA\Datasets\clima_piura_2000_2025_semanal.csv
